# 🔍 Clinisys Silver Layer Reconciliation: Local DuckDB vs AWS Athena Production

This notebook automates the data quality reconciliation between the local **DuckDB** database (`clinisys_all.duckdb`, schema `silver`) and the production **AWS Athena** database (`silver_clinisys_prod`).

### Objectives:
1. **Row Count Audit**: Compare total records per table.
2. **Primary Key Overlap Audit**: Reconcile exact keys present in both environments, only locally, or only in production.
3. **Yearly Breakdown**: Drill down row counts and key matches per calendar year (using mapped date columns).
4. **Newest Record Analysis**: Fetch and show the most recent records present in only one environment (ordered by primary key descending).

### Database Connections:
- **Local**: DuckDB (`clinisys_all.duckdb` -> schema: `silver`)
- **AWS Athena**: PyAthena (`silver_clinisys_prod` database)

In [1]:
import os
import yaml
import duckdb
import pandas as pd
import numpy as np
from pyathena import connect
import warnings
warnings.filterwarnings('ignore')

# Configuration
DUCKDB_PATH = '../../database/clinisys_all.duckdb'
ATHENA_REGION = 'sa-east-1'
ATHENA_WORKGROUP = 'datalake-admins'
ATHENA_DB = 'silver_clinisys_prod'
CONFIG_PATH = '../../clinisys/column_config.yml'

print("Libraries imported. Configured database paths:")
print(f"  Local DuckDB: {os.path.abspath(DUCKDB_PATH)}")
print(f"  AWS Athena:   {ATHENA_DB} (Region: {ATHENA_REGION}, Workgroup: {ATHENA_WORKGROUP})")

Libraries imported. Configured database paths:
  Local DuckDB: g:\My Drive\projetos_individuais\Huntington\database\clinisys_all.duckdb
  AWS Athena:   silver_clinisys_prod (Region: sa-east-1, Workgroup: datalake-admins)


## 🔌 Connection Helpers
Defining robust wrapper functions to connect, execute queries, and guarantee connection closure.

In [2]:
def run_duck(query):
    """Runs a query on local DuckDB, ensuring connection is closed."""
    conn = duckdb.connect(DUCKDB_PATH, read_only=True)
    try:
        return conn.execute(query).df()
    finally:
        conn.close()

def run_athena(query):
    """Runs a query on AWS Athena, ensuring connection is closed."""
    conn = connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB)
    try:
        return pd.read_sql(query, conn)
    finally:
        conn.close()

# Helper to get columns present in BOTH DuckDB and Athena schemas to prevent schema drift crashes
def get_common_columns(table):
    # 1. Local columns
    conn = duckdb.connect(DUCKDB_PATH, read_only=True)
    try:
        local_cols = set([c[0].lower() for c in conn.execute(f"SELECT * FROM silver.{table} LIMIT 0").description])
    finally:
        conn.close()
    
    # 2. Athena columns
    try:
        prod_df = run_athena(f"SELECT * FROM silver_clinisys_prod.{table} LIMIT 0")
        prod_cols = set([c.lower() for c in prod_df.columns])
    except Exception:
        prod_cols = set()
        
    return list(local_cols & prod_cols)

# Test connections
try:
    duck_ok = run_duck("SELECT 1 as test").iloc[0]['test'] == 1
    print("✅ DuckDB Connection: OK")
except Exception as e:
    print(f"❌ DuckDB Connection: FAILED - {e}")
    duck_ok = False

try:
    athena_ok = run_athena("SELECT 1 as test").iloc[0]['test'] == 1
    print("✅ AWS Athena Connection: OK")
except Exception as e:
    print(f"❌ AWS Athena Connection: FAILED - {e}")
    athena_ok = False

✅ DuckDB Connection: OK
✅ AWS Athena Connection: OK


## 🗺️ Configuration & Predefined Mappings
Loading primary keys from `column_config.yml` and defining mappings of tables to their primary date columns for yearly breakdowns.

In [3]:
# Load primary keys from configuration
with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)
primary_keys = config.get('primary_keys', {})

# Map tables to their primary date column (determined from analysis)
TABLE_DATE_COLUMNS = {
    'view_agenda': 'data',
    'view_agendas': 'data',
    'view_congelamentos_embrioes': 'data_congelamento',
    'view_congelamentos_ovulos': 'data_congelamento',
    'view_congelamentos_semen': 'data_congelamento',
    'view_congelamentos_semen_doador': 'data_congelamento',
    'view_descongelamentos_embrioes': 'data_descongelamento',
    'view_descongelamentos_ovulos': 'data_descongelamento',
    'view_embrioes_congelados': 'data_congelamento',
    'view_exames': 'data',
    'view_extrato_atendimentos_central': 'data',
    'view_indicacao_novo': 'data',
    'view_medicamentos_prescricoes': 'data_inicial',
    'view_micromanipulacao': 'data',
    'view_micromanipulacao_oocitos': 'data_procedimento',
    'view_orcamentos': 'data_entrega_orcamento',
    'view_ovulos_congelados': 'data_congelamento',
    'view_procedimentos_financas': 'data_pagamento',
    'view_tratamentos': 'data_procedimento',
    'view_tratamentos_us_anexos': 'data'
}

print(f"Loaded {len(primary_keys)} primary key configs.")
print(f"Predefined date columns for {len(TABLE_DATE_COLUMNS)} tables.")

Loaded 26 primary key configs.
Predefined date columns for 20 tables.


## 🔍 Identify Common Tables
Querying both databases to find matching tables in DuckDB `silver` schema and Athena `silver_clinisys_prod` database.

In [4]:
# Query tables
local_tables_df = run_duck("SELECT table_name FROM information_schema.tables WHERE table_schema = 'silver'")
local_tables = sorted(local_tables_df['table_name'].tolist())

athena_tables_df = run_athena("SHOW TABLES")
athena_tables = sorted(athena_tables_df.iloc[:, 0].tolist())

common_tables = sorted(list(set(local_tables) & set(athena_tables)))
print(f"Found {len(local_tables)} tables in local DuckDB 'silver' schema.")
print(f"Found {len(athena_tables)} tables in Athena '{ATHENA_DB}' database.")
print(f"Common tables to compare ({len(common_tables)}):")
for t in common_tables:
    key_status = f"Key: {primary_keys.get(t)}" if t in primary_keys else "⚠️ No Primary Key in config"
    
    # Schema-aware date column checking
    try:
        cols = get_common_columns(t)
        date_col = TABLE_DATE_COLUMNS.get(t)
        if date_col and date_col.lower() not in cols:
            fallbacks = [c for c in cols if 'data' in c or 'date' in c]
            date_col = fallbacks[0] if fallbacks else None
    except Exception:
        date_col = None
        
    date_status = f"Date: {date_col}" if date_col else "No Date column (All Time)"
    print(f"  - {t:<35} | {key_status:<25} | {date_status}")

Found 26 tables in local DuckDB 'silver' schema.
Found 27 tables in Athena 'silver_clinisys_prod' database.
Common tables to compare (26):
  - view_agenda                         | Key: id                   | Date: data
  - view_agendas                        | Key: id                   | No Date column (All Time)
  - view_congelamentos_embrioes         | Key: id                   | Date: responsavel_armazenamento_data
  - view_congelamentos_ovulos           | Key: id                   | Date: responsavel_armazenamento_data
  - view_congelamentos_semen            | Key: id                   | Date: responsavel_armazenamento_data
  - view_congelamentos_semen_doador     | Key: id                   | Date: responsavel_armazenamento_data
  - view_descongelamentos_embrioes      | Key: id                   | No Date column (All Time)
  - view_descongelamentos_ovulos        | Key: id                   | No Date column (All Time)
  - view_embrioes_congelados            | Key: id               

## 📊 Part 1: Row Count & Key Reconciliation Summary
Iterating through all common tables that have defined primary keys to perform count, match, and overlap checks.

In [5]:
summary_data = []

for t in common_tables:
    if t not in primary_keys:
        continue
    
    key = primary_keys[t]
    
    # Row counts
    local_count = run_duck(f"SELECT COUNT(*) as cnt FROM silver.{t}").iloc[0]['cnt']
    prod_count = run_athena(f"SELECT COUNT(*) as cnt FROM silver_clinisys_prod.{t}").iloc[0]['cnt']
    
    # Fetch keys to calculate overlap
    local_keys_df = run_duck(f"SELECT {key} FROM silver.{t}")
    local_keys_df.columns = [c.lower() for c in local_keys_df.columns]
    local_keys = set(local_keys_df[key.lower()].dropna().tolist())
    
    prod_keys_df = run_athena(f"SELECT {key} FROM silver_clinisys_prod.{t}")
    prod_keys_df.columns = [c.lower() for c in prod_keys_df.columns]
    prod_keys = set(prod_keys_df[key.lower()].dropna().tolist())
    
    matched_keys = len(local_keys & prod_keys)
    only_local = len(local_keys - prod_keys)
    only_prod = len(prod_keys - local_keys)
    
    summary_data.append({
        'Table': t,
        'Primary Key': key,
        'Local Rows': local_count,
        'Athena Rows': prod_count,
        'Difference': local_count - prod_count,
        'Match Count': matched_keys,
        'Only in Local (DuckDB)': only_local,
        'Only in Prod (Athena)': only_prod
    })

summary_df = pd.DataFrame(summary_data)
summary_df.style.format({
    'Local Rows': '{:,}',
    'Athena Rows': '{:,}',
    'Difference': '{:+,}',
    'Match Count': '{:,}',
    'Only in Local (DuckDB)': '{:,}',
    'Only in Prod (Athena)': '{:,}'
}).bar(subset=['Difference'], align='mid', color=['#d65f5f', '#5fba7d'])

,Table,Primary Key,Local Rows,Athena Rows,Difference,Match Count,Only in Local (DuckDB),Only in Prod (Athena)
0,view_agenda,id,"1,086,558","1,138,857","-52,299","1,086,543",15,"52,314"
1,view_agendas,id,262,262,+0,262,0,0
2,view_congelamentos_embrioes,id,"28,977","28,977",+0,"28,977",0,0
3,view_congelamentos_ovulos,id,"12,487","12,487",+0,"12,487",0,0
4,view_congelamentos_semen,id,"5,759","5,759",+0,"5,759",0,0
5,view_congelamentos_semen_doador,id,"1,978","1,978",+0,"1,978",0,0
6,view_descongelamentos_embrioes,id,"17,954","17,954",+0,"17,954",0,0
7,view_descongelamentos_ovulos,id,"4,065","4,065",+0,"4,065",0,0
8,view_embrioes_congelados,id,"95,852","95,852",+0,"95,852",0,0
9,view_exames,id,"37,150","37,150",+0,"37,150",0,0


## 📅 Part 2: Yearly Breakdown Analysis
Drilling down row counts and key overlaps per year for each table. For tables without defined date columns, counts are grouped under 'N/A'.

In [6]:
for t in common_tables:
    if t not in primary_keys:
        continue
        
    key = primary_keys[t]
    
    # Schema-aware date column extraction (common to both DBs)
    cols = get_common_columns(t)
    date_col = TABLE_DATE_COLUMNS.get(t)
    if date_col and date_col.lower() not in cols:
        fallbacks = [c for c in cols if 'data' in c or 'date' in c]
        date_col = fallbacks[0] if fallbacks else None
    
    print("=" * 80)
    print(f"📊 Table: {t} (Primary Key: {key} | Date Column: {date_col or 'N/A'})")
    print("=" * 80)
    
    # Fetch keys with date column
    if date_col:
        duck_q = f"SELECT {key} as key_val, {date_col} FROM silver.{t}"
        ath_q = f"SELECT {key} as key_val, {date_col} FROM silver_clinisys_prod.{t}"
        
        local_df = run_duck(duck_q)
        local_df.columns = [c.lower() for c in local_df.columns]
        
        prod_df = run_athena(ath_q)
        prod_df.columns = [c.lower() for c in prod_df.columns]
        
        # Normalize key and date column strings to lowercase
        key_lower = 'key_val'
        date_lower = date_col.lower()
        
        # Parse year safely using Pandas in Python to be 100% format-agnostic
        local_df['record_year'] = pd.to_datetime(local_df[date_lower], dayfirst=True, errors='coerce').dt.year
        local_df['record_year'] = local_df['record_year'].fillna('N/A').apply(lambda x: str(int(x)) if isinstance(x, (int, float)) and not pd.isna(x) else str(x))
        
        prod_df['record_year'] = pd.to_datetime(prod_df[date_lower], dayfirst=True, errors='coerce').dt.year
        prod_df['record_year'] = prod_df['record_year'].fillna('N/A').apply(lambda x: str(int(x)) if isinstance(x, (int, float)) and not pd.isna(x) else str(x))
    else:
        local_df = run_duck(f"SELECT {key} as key_val FROM silver.{t}")
        local_df.columns = [c.lower() for c in local_df.columns]
        local_df['record_year'] = 'N/A'
        
        prod_df = run_athena(f"SELECT {key} as key_val FROM silver_clinisys_prod.{t}")
        prod_df.columns = [c.lower() for c in prod_df.columns]
        prod_df['record_year'] = 'N/A'
        key_lower = 'key_val'
        
    # Unique years
    years = sorted(list(set(local_df['record_year'].dropna().tolist()) | set(prod_df['record_year'].dropna().tolist())))
    
    yearly_summary = []
    for yr in years:
        l_keys = set(local_df[local_df['record_year'] == yr][key_lower].dropna().tolist())
        p_keys = set(prod_df[prod_df['record_year'] == yr][key_lower].dropna().tolist())
        
        matched = len(l_keys & p_keys)
        only_l = len(l_keys - p_keys)
        only_p = len(p_keys - l_keys)
        
        yearly_summary.append({
            'Year': yr,
            'Local Count': len(l_keys),
            'Athena Count': len(p_keys),
            'Matched Count': matched,
            'Only Local': only_l,
            'Only Athena': only_p
        })
        
    yearly_df = pd.DataFrame(yearly_summary)
    display(yearly_df)
    print("\n")

📊 Table: view_agenda (Primary Key: id | Date Column: data)


,Year,Local Count,Athena Count,Matched Count,Only Local,Only Athena
0,1900,27,29,27,0,2
1,1920,1,1,1,0,0
2,1969,252,736,252,0,484
3,1970,2849,2849,2849,0,0
4,1997,1,1,1,0,0
5,2002,1319,1324,1319,0,5
6,2003,5020,5020,5020,0,0
7,2004,8564,8564,8564,0,0
8,2005,10576,10576,10576,0,0
9,2006,10393,10393,10393,0,0




📊 Table: view_agendas (Primary Key: id | Date Column: N/A)


,Year,Local Count,Athena Count,Matched Count,Only Local,Only Athena
0,N/A,262,262,262,0,0




📊 Table: view_congelamentos_embrioes (Primary Key: id | Date Column: responsavel_armazenamento_data)


...

,Year,Local Count,Athena Count,Matched Count,Only Local,Only Athena
0,2018,685,685,685,0,0
1,2019,19389,19389,19389,0,0
2,2020,17070,17070,17070,0,0
3,2021,22095,22095,22095,0,0
4,2022,23379,23379,23379,0,0
5,2023,45440,45440,45440,0,0
6,2024,61136,61136,61136,0,0
7,2025,100511,100511,100511,0,0
8,2026,83716,83717,83716,0,1




📊 Table: view_unidades (Primary Key: id | Date Column: N/A)


,Year,Local Count,Athena Count,Matched Count,Only Local,Only Athena
0,N/A,11,11,11,0,0




📊 Table: view_usuarios (Primary Key: id | Date Column: N/A)


,Year,Local Count,Athena Count,Matched Count,Only Local,Only Athena
0,N/A,1167,1167,1167,0,0


## 🆕 Part 3: Mismatch Drill-Down — Newest Mismatched Records
Fetching and showcasing up to 5 newest records (ordered by primary key descending) that are exclusive to either the Local database or AWS Athena database.

In [7]:
for t in common_tables:
    if t not in primary_keys:
        continue
        
    key = primary_keys[t]
    
    # Fetch keys from both
    local_keys_df = run_duck(f"SELECT {key} FROM silver.{t}")
    local_keys_df.columns = [c.lower() for c in local_keys_df.columns]
    local_keys = set(local_keys_df[key.lower()].dropna().tolist())
    
    prod_keys_df = run_athena(f"SELECT {key} FROM silver_clinisys_prod.{t}")
    prod_keys_df.columns = [c.lower() for c in prod_keys_df.columns]
    prod_keys = set(prod_keys_df[key.lower()].dropna().tolist())
    
    only_l_keys = list(local_keys - prod_keys)
    only_p_keys = list(prod_keys - local_keys)
    
    # Sort only keys descending
    only_l_keys.sort(reverse=True)
    only_p_keys.sort(reverse=True)
    
    print("=" * 80)
    print(f"🔍 Mismatch Samples: {t} (Primary Key: {key})")
    print("=" * 80)
    
    # Sample Local only
    if only_l_keys:
        sample_keys = only_l_keys[:5]
        keys_placeholder = ", ".join([str(k) if isinstance(k, (int, float)) else f"'{k}'" for k in sample_keys])
        sample_q = f"SELECT * FROM silver.{t} WHERE {key} IN ({keys_placeholder}) ORDER BY {key} DESC"
        local_samples = run_duck(sample_q)
        print(f"🆕 Top {len(sample_keys)} NEWEST records ONLY found in Local DuckDB (Total: {len(only_l_keys)}):")
        display(local_samples)
    else:
        print("✅ No records found exclusively in Local DuckDB.")
        
    # Sample Athena only
    if only_p_keys:
        sample_keys = only_p_keys[:5]
        keys_placeholder = ", ".join([str(k) if isinstance(k, (int, float)) else f"'{k}'" for k in sample_keys])
        sample_q = f"SELECT * FROM silver_clinisys_prod.{t} WHERE {key} IN ({keys_placeholder}) ORDER BY {key} DESC"
        prod_samples = run_athena(sample_q)
        print(f"🆕 Top {len(sample_keys)} NEWEST records ONLY found in Athena Production (Total: {len(only_p_keys)}):")
        display(prod_samples)
    else:
        print("✅ No records found exclusively in Athena Production.")
    print("\n")

🔍 Mismatch Samples: view_agenda (Primary Key: id)
🆕 Top 5 NEWEST records ONLY found in Local DuckDB (Total: 15):


,id,id_sms,id_laboratorio,data,inicio,termino,recorrente,recorrente_tipo,encaixe,evento,...,acompanhante_paciente,identificacao_visual_paciente,usuario_status,indicacao,rd_contato_id,rd_negociacao_id,link_consulta,hash,extraction_timestamp,is_deleted
0,1757703,0,0000045193,NaT,1900-01-01 07:00:00,07:15,0,None,None,216,...,None,None,None,None,<NA>,<NA>,None,5dc2f1317def94323c88e910c6a7c0ff,2026-07-28 19:11:56,0
1,1757702,0,0000045193,NaT,1900-01-01 07:00:00,07:15,0,None,None,216,...,None,None,None,None,<NA>,<NA>,None,709eead1dc59eaf77a108d939e278b41,2026-07-28 19:11:56,0
2,1757701,0,0000045193,NaT,1900-01-01 07:00:00,07:15,0,None,None,216,...,None,None,None,None,<NA>,<NA>,None,ba349baff8e4012a419f14b8c5d353db,2026-07-28 19:11:56,0
3,1757700,0,0000045193,NaT,1900-01-01 07:00:00,07:15,0,None,None,216,...,None,None,None,None,<NA>,<NA>,None,070221f30dfb0767800b2c44747d320b,2026-07-28 19:11:56,0
4,1757699,0,0000045193,NaT,1900-01-01 07:00:00,07:15,0,None,None,216,...,None,None,None,None,<NA>,<NA>,None,6b5965e01b1f23f79cab341234d7a64b,2026-07-28 19:11:56,0


🆕 Top 5 NEWEST records ONLY found in Athena Production (Total: 52314):


,id,id_sms,id_laboratorio,data,flag_date_suspect,inicio,termino,recorrente,recorrente_tipo,encaixe,...,tipo_atendimento,acompanhante_paciente,identificacao_visual_paciente,usuario_status,indicacao,rd_contato_id,rd_negociacao_id,link_consulta,bronze_updated_at,_dlt_id
0,1766461,0,0000027260,2024-06-06,False,00:00:00,00:15:00,0,None,None,...,None,None,None,None,None,None,None,None,2026-08-05 02:17:28.710,PgivplfayUYbzw
1,1766460,0,0000036968,2027-05-11,False,07:00:00,07:15:00,0,None,None,...,None,None,None,None,None,None,None,None,2026-08-05 02:17:28.710,XNkqO2MmaNNPUg
2,1766459,0,0000036968,2027-04-27,False,07:00:00,07:15:00,0,None,None,...,None,None,None,None,None,None,None,None,2026-08-05 02:17:28.710,cSUzJSZzUWc5sw
3,1766458,0,0000036968,2027-04-13,False,07:00:00,07:15:00,0,None,None,...,None,None,None,None,None,None,None,None,2026-08-05 02:17:28.710,r0kfn86TPXmtvA
4,1766457,0,0000036968,2027-03-16,False,07:00:00,07:15:00,0,None,None,...,None,None,None,None,None,None,None,None,2026-08-05 02:17:28.710,ashSG8Z1dsS9MQ




🔍 Mismatch Samples: view_agendas (Primary Key: id)
✅ No records found exclusively in Local DuckDB.
✅ No records found exclusively in Athena Production.


🔍 Mismatch Samples: view_congelamentos_embrioes (Primary Key: id)
✅ No records found exclusively in Local DuckDB.
✅ No records found exclusively in Athena Production.


🔍 Mismatch Samples: view_congelamentos_ovulos (Primary Key: id)
✅ No records found exclusively in Local DuckDB.
✅ No records found exclusively in Athena Production.


🔍 Mismatch Samples: view_congelamentos_semen (Primary Key: id)
✅ No records found exclusively in Local DuckDB.
✅ No records found exclusively in Athena Production.


🔍 Mismatch Samples: view_congelamentos_semen_doador (Primary Key: id)
✅ No records found exclusively in Local DuckDB.
✅ No records found exclusively in Athena Production.


🔍 Mismatch Samples: view_descongelamentos_embrioes (Primary Key: id)
✅ No records found exclusively in Local DuckDB.
✅ No records found exclusively in Athena Production.

...



🔍 Mismatch Samples: view_ovulos_congelados (Primary Key: id)
✅ No records found exclusively in Local DuckDB.
✅ No records found exclusively in Athena Production.


🔍 Mismatch Samples: view_pacientes (Primary Key: codigo)
✅ No records found exclusively in Local DuckDB.
✅ No records found exclusively in Athena Production.


🔍 Mismatch Samples: view_procedimentos (Primary Key: id)
✅ No records found exclusively in Local DuckDB.
✅ No records found exclusively in Athena Production.


🔍 Mismatch Samples: view_procedimentos_financas (Primary Key: id)
✅ No records found exclusively in Local DuckDB.
🆕 Top 5 NEWEST records ONLY found in Athena Production (Total: 75):


,id,procedimento,valor,duracao,exibir_indicacao,descricao,categoria,bronze_updated_at,_dlt_id
0,765451,Centoscreen,None,30,Sim,None,Procedimentos acessórios ao tratamento,2026-07-16 02:20:10.438,iffR1CkgG5Xayw
1,765450,Análise imunohistoquimica,None,30,Sim,None,Exames Gerais,2026-07-16 02:20:10.438,3a+jCRPeAZh3tg
2,765449,Inseminação Intra-Uterina Heteróloga,None,30,Sim,None,Tratamentos,2026-07-16 02:20:10.438,GblRbS4a/BZ8ag
3,765448,(FOT) Descongelamento de Óvulos Próprios + ICS...,None,30,Sim,None,Tratamentos,2026-07-16 02:20:10.438,+rsEAjfB1MLpAQ
4,765447,(FET Excedente) Descongelamento e Transferênci...,None,30,Sim,None,Tratamentos,2026-07-16 02:20:10.438,V2G28xvxkFHXBA




🔍 Mismatch Samples: view_tratamentos (Primary Key: id)
✅ No records found exclusively in Local DuckDB.
✅ No records found exclusively in Athena Production.


🔍 Mismatch Samples: view_tratamentos_us_anexos (Primary Key: id)
✅ No records found exclusively in Local DuckDB.
🆕 Top 1 NEWEST records ONLY found in Athena Production (Total: 1):


,id,prontuario,id_tratamento,data,data_us,nome_arquivo,arquivo,bronze_updated_at,_dlt_id
0,381142,895958,45927,2026-08-04,04/08/2026,CamScanner 04-08-2026 18.18.pdf,fba589b17fdc15a093efdae0417df246.pdf,2026-08-05 02:26:03.403,D/RgKz/1sVAczw




🔍 Mismatch Samples: view_unidades (Primary Key: id)
✅ No records found exclusively in Local DuckDB.
✅ No records found exclusively in Athena Production.


🔍 Mismatch Samples: view_usuarios (Primary Key: id)
✅ No records found exclusively in Local DuckDB.
✅ No records found exclusively in Athena Production.


